# Speaker Bio Card Generator (Refined)
Generate comprehensive character bio cards for each unique speaker from training data.
These cards will be used by the council to inform emotion classification decisions.

## Process:
1. Load training data and identify unique speakers
2. Sample 50-100 utterances per speaker for rich context
3. Generate bio cards using LLM with structured prompt
4. Save to `logs/speaker_bio_cards.txt` for integration with council
5. Parse and validate bio cards

In [9]:
import sys
import os
from groq import Groq
from dotenv import load_dotenv
import pandas as pd
import json
import re

# Setup paths and load environment
load_dotenv()
sys.path.append('../')
from src.load_data import load_data_from_csv

# Load API keys
BIO_CARD_API = os.getenv("BIO_CARD_GENERATOR")

if BIO_CARD_API:
    print(f"✓ API key loaded: {BIO_CARD_API[:20]}...")
else:
    raise ValueError("✗ BIO_CARD_GENERATOR API key not found in .env")

✓ API key loaded: gsk_9HXYjquJyK4Lse6n...


In [10]:
# Load and explore training data
training_data = load_data_from_csv('../data/train_sent_emo.csv')
training_data.drop_duplicates(inplace=True)

print(f"Loaded {len(training_data)} training samples")
print(f"\nUnique speakers: {training_data['Speaker'].nunique()}")
print(f"Speakers: {sorted(training_data['Speaker'].unique())}")
print(f"\nData shape: {training_data.shape}")
print(f"Columns: {training_data.columns.tolist()}")

Data loaded successfully from ../data/train_sent_emo.csv
Loaded 9989 training samples

Unique speakers: 260
Speakers: ['1st Customer', '2nd Customer', '3rd Customer', 'A Female Student', 'A Student', 'Airline Employee', 'Alice', 'All', 'Allesandro', 'Angela', 'Annabelle', 'Another Scientist', 'Another Tour Guide', 'Aunt Lillian', 'Barry', 'Ben', 'Bernice', 'Bob', 'Bobby', 'Bonnie', 'Both', 'Boy in the Cape', 'Burt', 'Caitlin', 'Carl', 'Carol', 'Casey', 'Cassie', 'Cecilia', 'Chandler', 'Charlie', 'Charlton Heston', 'Chip', 'Chloe', 'Cliff', 'Commercial', 'Customer', 'Dana', 'Danny', 'David', 'Dina', 'Director', 'Doctor', 'Doug', 'Dr. Baldhara', 'Dr. Drake Remoray', 'Dr. Franzblau', 'Dr. Green', 'Dr. Johnson', 'Dr. Ledbetter', 'Dr. Leedbetter', 'Dr. Long', 'Dr. Miller', 'Dr. Oberman', 'Dr. Rhodes', 'Dr. Stryker Ramoray', 'Dr. Wesley', 'Dr. Zane', 'Drunken Gambler', 'Duncan', 'Earl', 'Elizabeth', 'Emeril', 'Emily', 'Eric', 'Estelle', 'Evil Bitch', 'Fake Monica', 'Fireman No. 1', 'Fireman 

In [11]:
def groq_llm_call(prompt, model, api_key, max_tokens=2048):
    """Call Groq API for bio card generation."""
    try:
        client = Groq(api_key=api_key)
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model=model,
            temperature=1,
            max_tokens=max_tokens
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        raise Exception(f"Groq API Error: {str(e)}")

print("✓ groq_llm_call function defined")

✓ groq_llm_call function defined


In [12]:
# Extract rich sampled dialogues for each speaker (50-100 samples)
speaker_samples = {}

for speaker in sorted(training_data['Speaker'].unique()):
    speaker_data = training_data[training_data['Speaker'] == speaker]
    total_utterances = len(speaker_data)
    
    # Classify as generic character if less than 10 utterances
    if total_utterances < 10:
        speaker_samples[speaker] = {
            'total_utterances': total_utterances,
            'sample_count': total_utterances,
            'samples': speaker_data[['Utterance', 'Emotion', 'Sentiment']].to_dict(orient='records'),
            'emotion_distribution': speaker_data['Emotion'].value_counts().to_dict(),
            'sentiment_distribution': speaker_data['Sentiment'].value_counts().to_dict(),
            'character_type': 'GENERIC_CHARACTER'
        }
        print(f"\n{speaker}:")
        print(f"  Total utterances: {total_utterances}")
        print(f"  Classification: GENERIC_CHARACTER (< 10 utterances)")
        print(f"  Emotion dist: {speaker_data['Emotion'].value_counts().to_dict()}")
        print(f"  Sentiment dist: {speaker_data['Sentiment'].value_counts().to_dict()}")
    else:
        # Get 50-100 sample utterances for rich psychological context
        sample_size = min(max(50, len(speaker_data) // 2), min(100, len(speaker_data)))
        samples = speaker_data.sample(n=sample_size, random_state=42)[['Utterance', 'Emotion', 'Sentiment']]
        
        speaker_samples[speaker] = {
            'total_utterances': total_utterances,
            'sample_count': sample_size,
            'samples': samples.to_dict(orient='records'),
            'emotion_distribution': speaker_data['Emotion'].value_counts().to_dict(),
            'sentiment_distribution': speaker_data['Sentiment'].value_counts().to_dict(),
            'character_type': 'PRIMARY_CHARACTER'
        }
        
        print(f"\n{speaker}:")
        print(f"  Total utterances: {total_utterances}")
        print(f"  Sampled: {sample_size} utterances")
        print(f"  Classification: PRIMARY_CHARACTER")
        print(f"  Emotion dist: {speaker_data['Emotion'].value_counts().to_dict()}")
        print(f"  Sentiment dist: {speaker_data['Sentiment'].value_counts().to_dict()}")


1st Customer:
  Total utterances: 1
  Classification: GENERIC_CHARACTER (< 10 utterances)
  Emotion dist: {'joy': 1}
  Sentiment dist: {'positive': 1}

2nd Customer:
  Total utterances: 1
  Classification: GENERIC_CHARACTER (< 10 utterances)
  Emotion dist: {'joy': 1}
  Sentiment dist: {'positive': 1}

3rd Customer:
  Total utterances: 2
  Classification: GENERIC_CHARACTER (< 10 utterances)
  Emotion dist: {'neutral': 1, 'sadness': 1}
  Sentiment dist: {'neutral': 1, 'negative': 1}

A Female Student:
  Total utterances: 1
  Classification: GENERIC_CHARACTER (< 10 utterances)
  Emotion dist: {'neutral': 1}
  Sentiment dist: {'neutral': 1}

A Student:
  Total utterances: 1
  Classification: GENERIC_CHARACTER (< 10 utterances)
  Emotion dist: {'surprise': 1}
  Sentiment dist: {'negative': 1}

Airline Employee:
  Total utterances: 2
  Classification: GENERIC_CHARACTER (< 10 utterances)
  Emotion dist: {'neutral': 2}
  Sentiment dist: {'neutral': 2}

Alice:
  Total utterances: 3
  Classifi

In [13]:
# BIO CARD PROMPT TEMPLATE
# This template integrates the user's provided prompt with structured data

BIO_CARD_PROMPT_TEMPLATE = """
Role: Expert Social Psychologist & Character Profiler
Task: Analyze the provided dialogue samples to construct a "Static Persona Card" for the speaker.
This card will serve as the ground truth baseline for emotion recognition in the council system.

Instructions:
Based strictly on the provided conversation turns, define the following five dimensions:

1. **Core Personality**: Define their overarching temperament (e.g., neurotic, easy-going, sarcastic).
2. **Linguistic Signature**: Identify recurring speech patterns or rhetorical styles (e.g., self-deprecation, formal speech, high-energy outbursts).
3. **Arousal Baseline**: Is their "Neutral" state naturally high-energy (fast-talking/agitated) or low-energy (monotone/calm)?
4. **Conflict Style**: How do they typically express negative sentiment? (e.g., Does "Anger" manifest as direct aggression or as flustered defense/sarcasm?)
5. **Social Anchors**: Identify 1–2 key relationships and how their tone changes when interacting with those specific people.

INPUT DATA:
Speaker: {speaker_name}
Total Utterances in Training: {total_utterances}
Emotion Distribution: {emotion_distribution}
Sentiment Distribution: {sentiment_distribution}

DIALOGUE SAMPLES ({sample_count} turns):
{samples}

OUTPUT FORMAT (Strict JSON):
{{
  "speaker": "{speaker_name}",
  "static_persona": "[2-sentence summary of personality and role]",
  "linguistic_style": "[Key speech traits and idiosyncratic habits]",
  "baseline_arousal": "[Low/Medium/High and reasoning]",
  "negative_expression": "[How they express anger vs. sadness vs. sarcasm]",
  "social_relationship_priors": "[Behavioral notes when talking to others]"
}}
"""

print("✓ Bio card prompt template defined")

✓ Bio card prompt template defined


In [14]:
# Generate Bio Cards for All Speakers
bio_cards = {}
bio_cards_json = {}
output_file = "../logs/speaker_bio_cards.txt"
json_output_file = "../logs/speaker_bio_cards.json"

# Ensure logs directory exists
os.makedirs('../logs', exist_ok=True)

# Initialize output file with UTF-8 encoding
with open(output_file, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("SPEAKER BIO CARDS - Generated from Training Data\n")
    f.write(f"Total unique speakers: {len(speaker_samples)}\n")
    f.write("="*80 + "\n\n")

print(f"Generating bio cards for {len(speaker_samples)} speakers...\n")

for idx, (speaker_name, speaker_info) in enumerate(speaker_samples.items(), 1):
    character_type = speaker_info.get('character_type', 'PRIMARY_CHARACTER')
    print(f"[{idx}/{len(speaker_samples)}] Processing {speaker_name} ({character_type})...", end=" ")
    
    # Handle generic characters (skip LLM call)
    if character_type == 'GENERIC_CHARACTER':
        generic_bio = {
            "speaker": speaker_name,
            "character_type": "GENERIC_CHARACTER",
            "static_persona": "[Generic/Supporting character with limited dialogue]",
            "linguistic_style": "[Insufficient data]",
            "baseline_arousal": "[Unknown]",
            "negative_expression": "[Insufficient data]",
            "social_relationship_priors": "[Insufficient data]"
        }
        bio_cards[speaker_name] = json.dumps(generic_bio, indent=2, ensure_ascii=False)
        bio_cards_json[speaker_name] = generic_bio
        
        with open(output_file, 'a', encoding='utf-8') as f:
            f.write(f"\n{'='*80}\n")
            f.write(f"SPEAKER: {speaker_name}\n")
            f.write(f"Total Utterances: {speaker_info['total_utterances']}\n")
            f.write(f"Character Type: GENERIC_CHARACTER\n")
            f.write(f"{'='*80}\n\n")
            f.write(bio_cards[speaker_name])
            f.write("\n\n")
        
        print(f"✓")
        continue
    
    # Format samples with line numbers for clarity, skip utterances with encoding issues
    samples_list = []
    for i, s in enumerate(speaker_info['samples']):
        try:
            # Handle encoding issues by converting to string and replacing problematic chars
            utterance_text = str(s['Utterance'])[:120] if s['Utterance'] else "[EMPTY]"
            emotion_text = str(s['Emotion']) if s['Emotion'] else "[UNKNOWN]"
            sentiment_text = str(s['Sentiment']) if s['Sentiment'] else "[UNKNOWN]"
            samples_list.append(f"  {i+1}. Utterance: {utterance_text} |Emotion: {emotion_text} | Sentiment: {sentiment_text}")
        except Exception as e:
            # Skip utterances that cause encoding errors
            continue
    
    if not samples_list:
        print(f"✗ No valid utterances to process")
        continue
    
    samples_text = "\n".join(samples_list)
    
    # Build the full prompt
    full_prompt = BIO_CARD_PROMPT_TEMPLATE.format(
        speaker_name=speaker_name,
        total_utterances=speaker_info['total_utterances'],
        emotion_distribution=speaker_info['emotion_distribution'],
        sentiment_distribution=speaker_info['sentiment_distribution'],
        sample_count=speaker_info['sample_count'],
        samples=samples_text
    )
    
    try:
        # Call LLM to generate bio card
        bio_card_text = groq_llm_call(
            prompt=full_prompt,
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            api_key=BIO_CARD_API,
            max_tokens=2048
        )
        
        bio_cards[speaker_name] = bio_card_text
        
        # Try to parse as JSON for validation
        try:
            json_match = re.search(r'\{.*\}', bio_card_text, re.DOTALL)
            if json_match:
                bio_cards_json[speaker_name] = json.loads(json_match.group())
        except:
            bio_cards_json[speaker_name] = {"raw": bio_card_text}
        
        # Write to text file with UTF-8 encoding
        with open(output_file, 'a', encoding='utf-8') as f:
            f.write(f"\n{'='*80}\n")
            f.write(f"SPEAKER: {speaker_name}\n")
            f.write(f"Total Utterances: {speaker_info['total_utterances']}\n")
            f.write(f"Sampled: {speaker_info['sample_count']} utterances\n")
            f.write(f"Character Type: PRIMARY_CHARACTER\n")
            f.write(f"{'='*80}\n\n")
            f.write(bio_card_text)
            f.write("\n\n")
        
        print(f"✓")
    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}...")
        bio_cards[speaker_name] = f"ERROR: {str(e)}"
        bio_cards_json[speaker_name] = {"error": str(e)}
        with open(output_file, 'a', encoding='utf-8') as f:
            f.write(f"\nERROR generating bio card for {speaker_name}: {str(e)}\n\n")

# Save JSON version with UTF-8 encoding
with open(json_output_file, 'w', encoding='utf-8') as f:
    json.dump(bio_cards_json, f, indent=2, ensure_ascii=False)

print(f"\n✓ Bio cards saved to:")
print(f"  - Text: {os.path.abspath(output_file)}")
print(f"  - JSON: {os.path.abspath(json_output_file)}")

Generating bio cards for 260 speakers...

[1/260] Processing 1st Customer (GENERIC_CHARACTER)... ✓
[2/260] Processing 2nd Customer (GENERIC_CHARACTER)... ✓
[3/260] Processing 3rd Customer (GENERIC_CHARACTER)... ✓
[4/260] Processing A Female Student (GENERIC_CHARACTER)... ✓
[5/260] Processing A Student (GENERIC_CHARACTER)... ✓
[6/260] Processing Airline Employee (GENERIC_CHARACTER)... ✓
[7/260] Processing Alice (GENERIC_CHARACTER)... ✓
[8/260] Processing All (PRIMARY_CHARACTER)... ✓
[9/260] Processing Allesandro (GENERIC_CHARACTER)... ✓
[10/260] Processing Angela (GENERIC_CHARACTER)... ✓
[11/260] Processing Annabelle (GENERIC_CHARACTER)... ✓
[12/260] Processing Another Scientist (GENERIC_CHARACTER)... ✓
[13/260] Processing Another Tour Guide (GENERIC_CHARACTER)... ✓
[14/260] Processing Aunt Lillian (GENERIC_CHARACTER)... ✓
[15/260] Processing Barry (PRIMARY_CHARACTER)... ✓
[16/260] Processing Ben (PRIMARY_CHARACTER)... ✓
[17/260] Processing Bernice (GENERIC_CHARACTER)... ✓
[18/260] Proc

In [15]:
# Summary and Validation
print("\nBIO CARD GENERATION SUMMARY")
print("="*70)

success_count = 0
error_count = 0

for speaker_name, bio_card in bio_cards.items():
    if bio_card.startswith("ERROR"):
        print(f"✗ {speaker_name}: {bio_card[:40]}...")
        error_count += 1
    else:
        print(f"✓ {speaker_name}: {len(bio_card)} characters")
        success_count += 1

print("="*70)
print(f"\nGenerated: {success_count} bio cards")
print(f"Errors: {error_count} bio cards")
print(f"Success Rate: {100 * success_count / len(bio_cards):.1f}%")


BIO CARD GENERATION SUMMARY
✓ 1st Customer: 331 characters
✓ 2nd Customer: 331 characters
✓ 3rd Customer: 331 characters
✓ A Female Student: 335 characters
✓ A Student: 328 characters
✓ Airline Employee: 335 characters
✓ Alice: 324 characters
✓ All: 1869 characters
✓ Allesandro: 329 characters
✓ Angela: 325 characters
✓ Annabelle: 328 characters
✓ Another Scientist: 336 characters
✓ Another Tour Guide: 337 characters
✓ Aunt Lillian: 331 characters
✓ Barry: 2051 characters
✓ Ben: 1956 characters
✓ Bernice: 326 characters
✓ Bob: 322 characters
✓ Bobby: 324 characters
✓ Bonnie: 325 characters
✓ Both: 323 characters
✓ Boy in the Cape: 334 characters
✓ Burt: 323 characters
✓ Caitlin: 326 characters
✓ Carl: 323 characters
✓ Carol: 1993 characters
✓ Casey: 324 characters
✓ Cassie: 2350 characters
✓ Cecilia: 326 characters
✓ Chandler: 1683 characters
✓ Charlie: 2556 characters
✓ Charlton Heston: 334 characters
✓ Chip: 1630 characters
✓ Chloe: 324 characters
✓ Cliff: 324 characters
✓ Commercia

In [16]:
# Utility: Load bio cards for council integration
def load_bio_cards(json_file="../logs/speaker_bio_cards.json"):
    """Load bio cards from JSON file for use in council decision-making."""
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"✗ Bio cards file not found at {json_file}")
        return {}
    except json.JSONDecodeError:
        print(f"✗ Error decoding JSON from {json_file}")
        return {}

# Test loading
loaded_cards = load_bio_cards()
print(f"\n✓ Loaded {len(loaded_cards)} bio cards for council integration")

# Show sample
if loaded_cards:
    first_speaker = list(loaded_cards.keys())[0]
    print(f"\nSample (First Speaker): {first_speaker}")
    print(json.dumps(loaded_cards[first_speaker], indent=2, ensure_ascii=False)[:400])


✓ Loaded 260 bio cards for council integration

Sample (First Speaker): 1st Customer
{
  "speaker": "1st Customer",
  "character_type": "GENERIC_CHARACTER",
  "static_persona": "[Generic/Supporting character with limited dialogue]",
  "linguistic_style": "[Insufficient data]",
  "baseline_arousal": "[Unknown]",
  "negative_expression": "[Insufficient data]",
  "social_relationship_priors": "[Insufficient data]"
}
